# 07. Synthetic Evaluation Dataset Generation

This notebook demonstrates synthetic test-set generation using OpenAI Structured Outputs and Pydantic schema validation (`src/step7_make_dataset.py`).

### Key Demonstration Steps:
1. **Pydantic Schema Enforcement**: Defining `EvaluationItem` to ensure structured, typed benchmark fields (`question`, `keywords`, `reference_answer`, `category`).
2. **Category Round-Robin Distribution**: Sampling across 5 distinct evaluation categories (*parables*, *named entities*, *definitions*, *praxis*, *quote grounding*).
3. **Concurrent Generation Preview**: Executing a 2-file test batch using `ThreadPoolExecutor` and inspecting output schema compliance.

In [1]:
import sys
import os

# Add project root to path for modular imports
sys.path.append("..")

from src.step7_make_dataset import (
    EvaluationItem, 
    CATEGORIES, 
    process_file,
    PROCESSED_DATA_DIR
)

## Step 1: Pydantic Schema & Category Definitions

Inspect the structured data schema used by `client.beta.chat.completions.parse` to ensure deterministic synthetic outputs.

In [2]:
print("--- Synthetic Test Schema Fields ---")
for field_name, field_info in EvaluationItem.model_fields.items():
    print(f"• {field_name:18s} : {field_info.description}")

print("\n--- Target Evaluation Categories ---")
for cat in CATEGORIES:
    print(f"• {cat}")

--- Synthetic Test Schema Fields ---
• source_file        : The source filename, e.g., 'Gadhada_I_1.md'
• question           : A specific, grounded question based ONLY on the text snippet.
• keywords           : 3 to 5 key terms present in the text.
• reference_answer   : Concise, factual ground-truth answer derived strictly from the text.
• category           : The target category assigned.

--- Target Evaluation Categories ---
• exact_analogy_or_parable
• named_entities_and_location
• explicit_definition_or_classification
• cause_and_effect_praxis
• textual_quote_grounding


## Step 2: Sample Test Case Generation

Execute `process_file` on a sample Markdown document to inspect generated synthetic questions and ground-truth reference answers.

In [3]:
sample_file = list((PROCESSED_DATA_DIR / "vachanamruts").rglob("*.md"))[0]
target_category = "cause_and_effect_praxis"

print(f"Processing File : {sample_file.name}")
print(f"Category Target : {target_category}\n")

result = process_file((sample_file, target_category))

print("--- Generated Evaluation Object ---")
print(f"Source File  : {result['source_file']}")
print(f"Category     : {result['category']}")
print(f"Question     : {result['question']}")
print(f"Ref Answer   : {result['reference_answer']}")
print(f"Keywords     : {', '.join(result['keywords'])}")

Processing File : Additional.md
Category Target : cause_and_effect_praxis

✅ Generated [cause_and_effect_praxis] for: Additional
--- Generated Evaluation Object ---
Source File  : Gadhada_I_1.md
Category     : cause_and_effect_praxis
Question     : What does the text suggest about the rarity of attaining a human birth in Bharat-khand and its significance for liberation?
Ref Answer   : The text states that attaining a human birth in Bharat-khand is extremely rare and is equated to receiving a chintamani. It emphasizes that even deities like Indra long for a human birth because only in Bharat-khand can one attain liberation, making it superior to being born in any other region of Mrutyulok.
Keywords     : human birth, Bharat-khand, liberation, rarity, deities


## Step 3: Dataset File Target Path Verification

Verify output path resolution for `tests.jsonl` within `PROCESSED_DATA_DIR`.

In [4]:
output_jsonl = PROCESSED_DATA_DIR / "tests.jsonl"
print(f"Target Evaluation Dataset Path: {output_jsonl}")

Target Evaluation Dataset Path: C:\Users\Lenovo\projects\Active Vachanamrut RAG project\data\processed\tests.jsonl
